# Config & Setup

In [1]:
# ========== CONFIGURATION ==========
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
import torchvision.transforms as T
import torchvision.models as models

import pandas as pd
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

# ========== PATHS (KAGGLE) ==========
DATA_ROOT = Path("/kaggle/input/chexpert")
TRAIN_CSV = DATA_ROOT / "train.csv"
VALID_CSV = DATA_ROOT / "valid.csv"
IMAGE_ROOT = DATA_ROOT


# Output
OUTPUT_DIR = Path("/kaggle/working")
CHECKPOINT_PATH = OUTPUT_DIR / "best_model.pth"

# ========== MODEL CONFIG ==========
MODEL_NAME = "resnet50"
NUM_CLASSES = 14
IMAGE_SIZE = 224
PRETRAINED = True

# ========== TRAINING CONFIG ==========
BATCH_SIZE = 64        # Increase to 64 if GPU memory allows
EPOCHS = 10             # Change to 10-15 for full training
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ========== LABELS ==========
LABEL_NAMES = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly",
    "Lung Opacity", "Lung Lesion", "Edema", "Consolidation",
    "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]

UNCERTAIN_POLICY = "zeros"

print(f"✓ Device: {DEVICE}")
print(f"✓ Data: {DATA_ROOT}")
print(f"✓ Training for {EPOCHS} epochs")


✓ Device: cuda
✓ Data: /kaggle/input/chexpert
✓ Training for 10 epochs


#  Dataset Class

In [2]:
class CheXpertDataset(Dataset):
    def __init__(self, csv_file, image_root, transform=None, uncertain_policy="zeros"):
        self.df = pd.read_csv(csv_file)
        self.image_root = Path(image_root)
        self.transform = transform
        self._process_labels(uncertain_policy)
        print(f"✓ Loaded {len(self.df)} samples")

    def _process_labels(self, policy):
        self.df[LABEL_NAMES] = self.df[LABEL_NAMES].fillna(0)
        if policy == "zeros":
            self.df[LABEL_NAMES] = self.df[LABEL_NAMES].replace(-1, 0)
        elif policy == "ones":
            self.df[LABEL_NAMES] = self.df[LABEL_NAMES].replace(-1, 1)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # FIX: Remove "CheXpert-v1.0-small/" prefix from path
        raw_path = row["Path"]
        if raw_path.startswith("CheXpert-v1.0-small/"):
            relative_path = raw_path.replace("CheXpert-v1.0-small/", "")
        else:
            relative_path = raw_path
        
        img_path = self.image_root / relative_path
        
        # Safety check
        if not img_path.exists():
            raise FileNotFoundError(f"Image not found: {img_path}")
        
        image = Image.open(img_path).convert("RGB")
        labels = torch.tensor(row[LABEL_NAMES].values.astype(np.float32))
        
        if self.transform:
            image = self.transform(image)
        
        return image, labels


def get_transforms(train=True):
    if train:
        return T.Compose([
            T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(10),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    else:
        return T.Compose([
            T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])


# Create dataloaders
train_ds = CheXpertDataset(TRAIN_CSV, IMAGE_ROOT, get_transforms(True), UNCERTAIN_POLICY)
val_ds = CheXpertDataset(VALID_CSV, IMAGE_ROOT, get_transforms(False), UNCERTAIN_POLICY)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Val batches: {len(val_loader)}")


✓ Loaded 223414 samples
✓ Loaded 234 samples
✓ Train batches: 3491
✓ Val batches: 4


# Model

In [3]:
class CheXpertModel(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.classifier = nn.Linear(in_features, NUM_CLASSES)

    def forward(self, x):
        feats = self.backbone(x)
        logits = self.classifier(feats)
        return logits


model = CheXpertModel().to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

print(f"✓ Model built: {MODEL_NAME} on {DEVICE}")


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 177MB/s]


✓ Model built: resnet50 on cuda


# Training Loop

In [4]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    pbar = tqdm(loader, desc="Training")
    
    for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        logits = model(images)
        loss = criterion(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(loader)


def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            all_preds.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    
    return avg_loss, all_preds, all_labels


# Train

In [5]:
print("="*60)
print(f"Starting training for {EPOCHS} epochs")
print("="*60 + "\n")

best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 40)
    
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_preds, val_labels = validate(model, val_loader, criterion)
    
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
        }, CHECKPOINT_PATH)
        print(f"✓ Saved best model (Val Loss: {val_loss:.4f})")

print("\n" + "="*60)
print("Training complete!")
print(f"Best Val Loss: {best_val_loss:.4f}")
print("="*60)


Starting training for 10 epochs


Epoch 1/10
----------------------------------------


Validation: 100%|██████████| 4/4 [00:01<00:00,  2.15it/s]


Train Loss: 0.2923 | Val Loss: 0.3804
✓ Saved best model (Val Loss: 0.3804)

Epoch 2/10
----------------------------------------


Validation: 100%|██████████| 4/4 [00:01<00:00,  3.90it/s]


Train Loss: 0.2755 | Val Loss: 0.3848

Epoch 3/10
----------------------------------------


...

Train Loss: 0.2556 | Val Loss: 0.3888

Epoch 9/10
----------------------------------------


Validation: 100%|██████████| 4/4 [00:01<00:00,  3.79it/s]


Train Loss: 0.2534 | Val Loss: 0.3811

Epoch 10/10
----------------------------------------


Validation: 100%|██████████| 4/4 [00:01<00:00,  3.77it/s]

Train Loss: 0.2510 | Val Loss: 0.3949

Training complete!
Best Val Loss: 0.3743


# Evaluate

In [6]:
def compute_metrics(y_true, y_pred):
    auroc_list, f1_list = [], []
    
    for i in range(NUM_CLASSES):
        if y_true[:, i].sum() == 0:
            continue
        try:
            auroc = roc_auc_score(y_true[:, i], y_pred[:, i])
        except:
            auroc = 0.5
        
        y_pred_bin = (y_pred[:, i] >= 0.5).astype(int)
        f1 = f1_score(y_true[:, i], y_pred_bin, zero_division=0)
        
        auroc_list.append(auroc)
        f1_list.append(f1)
    
    return {
        'auroc_mean': np.mean(auroc_list),
        'f1_mean': np.mean(f1_list)
    }


metrics = compute_metrics(val_labels, val_preds)
print("\n" + "="*60)
print("EVALUATION METRICS")
print("="*60)
print(f"AUROC:    {metrics['auroc_mean']:.4f}")
print(f"F1 Score: {metrics['f1_mean']:.4f}")
print("="*60)



EVALUATION METRICS
AUROC:    0.8027
F1 Score: 0.2917


# Download Model

In [7]:
# Model saved at: /kaggle/working/best_model.pth
# Click "Output" tab on right → Download best_model.pth
print(f"Model saved at: {CHECKPOINT_PATH}")
print("Download 'best_model.pth'")


Model saved at: /kaggle/working/best_model.pth
Download 'best_model.pth'
